In [ ]:
# ------------------------------
# IR System with AND/OR/Phrase + TF-IDF
# ------------------------------

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import glob
import math
from collections import Counter


In [ ]:
# ------------------------------
# 0. NLTK setup (first time)
# ------------------------------
!pip install nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def preprocess(text, do_lemmatize=False):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    if do_lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

In [ ]:
# ------------------------------
# 3. Compute IDF
# ------------------------------
def compute_idf(docs):
    N = len(docs)
    df = {}
    for doc in docs:
        tokens = set(preprocess(doc))
        for token in tokens:
            df[token] = df.get(token, 0) + 1
    idf = {term: math.log(N / df[term]) for term in df}
    return idf

In [ ]:
# ------------------------------
# 4. TF-IDF Ranking
# ------------------------------
def tfidf_ranking(query, docs, doc_ids, idf):
    q_tokens = preprocess(query)
    scores = {}
    for doc_id in doc_ids:
        tokens = preprocess(docs[doc_id])
        tf = {}
        for t in tokens:
            tf[t] = tf.get(t, 0) + 1
        score = 0
        for t in q_tokens:
            score += tf.get(t, 0) * idf.get(t, 0)
        scores[doc_id] = score
    ranked = sorted(doc_ids, key=lambda x: scores[x], reverse=True)
    return [(i, scores[i]) for i in ranked] # Changed to return (doc_id, score)

In [ ]:
# ------------------------------
# 5. Search with Positions
# ------------------------------
def search_with_positions(query, docs, index, mode="AND", phrase=False, use_tfidf=False, idf=None):
    q_tokens = preprocess(query)
    candidate_docs_with_info = [] # Stores (doc_id, doc_content, positions)

    for doc_id, doc in enumerate(docs):
        doc_tokens = preprocess(doc)

        # AND / OR
        if mode.upper() == "AND" and not all(t in doc_tokens for t in q_tokens):
            continue
        if mode.upper() == "OR" and not any(t in doc_tokens for t in q_tokens):
            continue

        # Phrase search
        if phrase:
            found = False
            for i in range(len(doc_tokens) - len(q_tokens) + 1):
                if doc_tokens[i:i+len(q_tokens)] == q_tokens:
                    found = True
                    break
            if not found:
                continue

        # Positions لكل كلمة
        positions = {t: index.get(t, {}).get(doc_id, []) for t in q_tokens}

        candidate_docs_with_info.append((doc_id, doc, positions))

    final_results = []
    if use_tfidf and idf:
        doc_ids_for_tfidf = [item[0] for item in candidate_docs_with_info]
        ranked_scores = tfidf_ranking(query, docs, doc_ids_for_tfidf, idf) # Returns (doc_id, score) tuples

        # Create a map for quick lookup of content and positions
        content_positions_map = {item[0]: (item[1], item[2]) for item in candidate_docs_with_info}

        for doc_id, score in ranked_scores:
            content, positions = content_positions_map.get(doc_id, ("", {})) # Fallback if doc_id somehow missing
            final_results.append((doc_id, content, positions, score))
    else:
        # If not using TF-IDF, add candidates with a None score
        for doc_id, content, positions in candidate_docs_with_info:
            final_results.append((doc_id, content, positions, None))

    return final_results # Now always returns (doc_id, content, positions, score)

In [ ]:
# ------------------------------
# 6. Load Documents (Upload)
# ------------------------------
docs = []
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    with open(filename, "r", encoding="utf-8") as f:
        docs.append(f.read())

print(f"{len(docs)} document(s) loaded successfully!")


Saving doc4.txt to doc4 (1).txt
Saving doc3.txt to doc3 (1).txt
Saving doc2.txt to doc2 (1).txt
Saving doc1.txt to doc1 (1).txt
4 document(s) loaded successfully!


In [ ]:
# ------------------------------
# 7. Build Index and IDF
# ------------------------------
index = build_index(docs)
idf = compute_idf(docs)

In [ ]:
# ------------------------------
# 8. Demo
# ------------------------------
print("Index built! Try queries like: 'information retrieval', 'search engines', 'natural language'")

while True:
    query = input("\nSearch (type 'quit' to exit): ").strip()
    if query.lower() == "quit":
        break
    mode = input("Mode (AND/OR): ").strip().upper()
    phrase = input("Phrase search? (y/n): ").strip().lower() == 'y'
    use_tfidf = input("Use TF-IDF ranking? (y/n): ").strip().lower() == 'y'

    results = search_with_positions(query, docs, index, mode=mode, phrase=phrase, use_tfidf=use_tfidf, idf=idf)

    print(f"\nFound {len(results)} document(s):")
    for doc_id, content, positions in results:
        print(f"[{doc_id}] {content}")
        print(f"Positions: {positions}\n")

Index built! Try queries like: 'information retrieval', 'search engines', 'natural language'

Search (type 'quit' to exit): information retrieval
Mode (AND/OR): OR
Phrase search? (y/n): y
Use TF-IDF ranking? (y/n): y

Found 2 document(s):
[3] Information retrieval is the process of obtaining relevant information from large collections of data. 
It involves searching, indexing, and ranking documents to answer user queries effectively.

Positions: {'information': [0, 5], 'retrieval': [1]}

[1] TF-IDF is a statistical measure used to evaluate how important a word is to a document in a collection. 
It is commonly used in information retrieval and text mining tasks.

Positions: {'information': [11], 'retrieval': [12]}



KeyboardInterrupt: Interrupted by user